# Tidal energy resource assessment

This notebook is part of a workflow perform a quality-checked flow statistics, tidal cycles,
vertical profiles, resolved variability, and an illustrative energy estimate using ADCP current measurements before performing a high-fidelity hydrodynamic modeling.

**Start here:** select the **Python (resource-assesment)** kernel, edit the JSON settings file, run Setup,
then run **Preprocess → Process → Postprocess** with **Shift+Enter**. For the bundled CO-OPS example,
**Run All** works with the defaults. A fresh run does not overwrite earlier results.

| Step | What you do | What you get |
|---|---|---|
| Preprocess | Select data, units/metadata and quality settings | Source-independent velocity table, coverage and bin inventory |
| Process | Run the shared analyses | Phases, cycles, profiles, residual variability and optional AEP |
| Postprocess | Save and explore | CSV tables, figures and a reproducibility run log |

The example flood direction, turbine and heading corrections are **site-specific**, not universal defaults
for other ADCP deployments. This is a resource-analysis workflow, not a certification of IEC compliance.

## Setup

Install the environment once from a terminal in this repository:

```text
conda env create -f environment.yml
conda activate resource-assesment
python -m ipykernel install --user --name resource-assesment --display-name "Python (resource-assesment)"
jupyter lab Tidal_Resource_Assessment.ipynb
```

For an existing environment, use `conda env update -n resource-assesment -f environment.yml`.
All imports below are local; the notebook does not fetch data or change the source files.

In [2]:
from pathlib import Path
from IPython.display import display
from backend.tidal import Settings, load_settings, preprocess, process, postprocess
from backend.tidal.notebook import show_preprocessing, show_results, explore

ROOT = Path.cwd()
if not (ROOT / "inputs").is_dir() or not (ROOT / "backend" / "tidal").is_dir():
    raise RuntimeError("Open this notebook from the repository root, alongside inputs/ and backend/.")
print("Ready. Edit your JSON settings file, then load it below.")

Ready. Edit your JSON settings file, then load it below.


### Choose the data and settings

Edit **config.json**, alongside this notebook. Settings are grouped into
`data_and_quality`, `flow_cycles_profiles`, `turbine_and_power`, and `comparisons`.
See **CONFIGURATION.md** for every field,
units and source-switching instructions. Changing `source` alone does not reset other fields.
Set total water depth in `data_and_quality.water_depth_m` (metres): Verdant Station A `24.0`,
Station B `26.9`, CO-OPS LIS1016 `37.8` (124 ft at Cornfield Point, 4 miles south of Buoy CF).
For another site, put the confirmed ADCP depth in that same field, or `null` if unknown.

- **CO-OPS:** put depth-bin observations in `inputs/coops/depth_bins/`. Filenames use
  `SITE_05m76cm-YYYY-MM-DD.csv`; any number of segments per depth is supported. The site name must
  match the filename prefix. Units and true bearings come from column headers.
- **Verdant Power:** the example workbooks are in `inputs/verdant/`. Sheet names and heading offsets
  correspond to alphabetically sorted workbooks. Defaults select Station B/B-2 and −5.53°/+3.20°.
  Both vector components are rotated consistently. Times use the workbook UTC column;
  heights come from its distance-from-bed row. The bundled B/B-2 preset excludes bins 45–49,
  which have poor coverage, and retains the complete first 44 bins. Review this explicit exclusion
  for other deployments; it is not a universal ADCP rule.
- **Other ADCP:** use the documented long CSV format in `inputs/custom/README.md`.
  Convert your instrument export once, then use exactly the same analyses. Arbitrary manufacturer
  formats need an adapter; the workflow does not guess their units or layout.

**Metadata matters:** CO-OPS filename depths do not establish the vertical datum. The default is
unconfirmed; the code never infers bed position from the velocity gradient. Provide a confirmed datum
and total water depth if available. A `null` water depth leaves calculations that require it unestimated.

Annual CO-OPS **event predictions** are retained in `inputs/coops/annual/` for reference. They are not
uniformly sampled ADCP measurements and are not sent through the turbulence/AEP pipeline.

Edit numbers without quotes; JSON uses `true`, `false`, and `null`. Comments and trailing commas are not allowed.

In [3]:
CONFIG_FILE = "config.json"
settings = load_settings(ROOT, CONFIG_FILE)
display(settings.to_dict())
print(f"Loaded {CONFIG_FILE}. Slack threshold = cut-in = {settings.cut_in_mps:g} m/s.")

{'source': 'coops',
 'site': 'LIS1016',
 'input_path': 'inputs/coops/depth_bins',
 'time_zone': 'unspecified',
 'coops_depth_reference': 'unknown',
 'verdant_sheets': ('Station B', 'Station B-2'),
 'verdant_heading_offsets': (-5.53, 3.2),
 'excluded_bins': (),
 'max_speed_mps': 6.0,
 'min_bin_coverage': 1.0,
 'axis_method': 'pca',
 'flood_reference_deg': 252.38,
 'slack_mps': 0.4,
 'smooth_samples': 3,
 'gap_factor': 1.5,
 'min_cycle_hours': 6.0,
 'max_cycle_hours': 18.0,
 'ensemble_minutes': 60,
 'noise_std_mps': 0.0,
 'water_depth_m': None,
 'near_bed_fraction': 0.2,
 'hub_bin': 13,
 'power_curve': 'inputs/turbine/power_curve_generated.csv',
 'rated_kw': 110.0,
 'rated_speed_mps': 1.5,
 'power_curve_mode': 'edited',
 'power_curve_points': 'speed_mps,power_kw\n0,0\n0.4,0\n1.5,110\n3.2,110',
 'cut_in_mps': 0.4,
 'cut_out_mps': 3.2,
 'speed_bin_width': 0.1,
 'turbine_count': 24,
 'availability': 1.0,
 'energy_enabled': True,
 'characteristic_length_m': 32.1177,
 'kinematic_viscosity_m2p

Loaded config.json. Slack threshold = cut-in = 0.4 m/s.


## 1. Preprocess — load, normalize and review quality

This step converts velocities to **m/s**, preserves true compass bearings, checks timestamps and
depth mappings, removes identical overlapping records, and flags missing/extreme velocities.
Conflicting overlapping records stop with a clear message so you can resolve their provenance.

Bins are aligned by actual timestamps, not row position. The default requires every selected bin
to be valid for a depth average; you can relax coverage explicitly. Averages are arithmetic means
of east/north components over measured bins, not integrals over the entire water column.

Review the inventory below. Confirm that the energy **hub bin** exists, the coordinate datum is correct,
and the sample interval matches the deployment. A missing velocity remains missing, never slack.
Settings are frozen for this assessment: after editing the JSON file, rerun this step and the steps below.

In [4]:
settings = load_settings(ROOT, CONFIG_FILE)  # Reload saved JSON edits on every preprocessing run.
assessment = preprocess(ROOT, settings)
show_preprocessing(assessment)

,source,site,start,end,time_zone,bins,timestamps,sample_minutes,valid_velocity_percent
0,coops,LIS1016,2010-06-09 20:12:00,2010-07-27 18:30:00,unspecified,17,11504,6.0,100.0


,input_rows,rows_without_time_bin_depth,identical_duplicates_removed,invalid_velocity_rows,retained_rows,retained_bins
0,195558,0,0,0,195558,17


,bin,depth_m,depth_reference,height_above_bed_m
0,1,5.76,unknown,NaN
1,2,7.77,unknown,NaN
2,3,9.78,unknown,NaN
3,4,11.77,unknown,NaN
4,5,13.78,unknown,NaN
5,6,15.76,unknown,NaN
6,7,17.77,unknown,NaN
7,8,19.78,unknown,NaN
8,9,21.76,unknown,NaN
9,10,23.77,unknown,NaN


,bin,samples,valid_samples,valid_fraction
0,1,11503,11503,1.0
1,2,11503,11503,1.0
2,3,11503,11503,1.0
3,4,11503,11503,1.0
4,5,11503,11503,1.0
5,6,11503,11503,1.0
6,7,11503,11503,1.0
7,8,11503,11503,1.0
8,9,11503,11503,1.0
9,10,11503,11503,1.0


• CO-OPS timestamps retain the downloaded clock; its timezone is unspecified. Confirm it before external time comparisons.
• CO-OPS depth datum is unconfirmed. Depth profiles are available; physical log/power fits require a confirmed height above bed.
• 1 depth-averaged timestamps fail the selected bin-coverage requirement.
• Depth averages are arithmetic vector means over retained valid bins, not full-column integrals.


## 2. Process — use one analysis pipeline

**Flow and phases.** PCA (or doubled-angle averaging) identifies a bidirectional channel axis.
Your floodward reference resolves its 180° ambiguity. All modules use positive along-channel velocity
for flood and negative for ebb. A centered moving average and the slack threshold produce one
timestamp-level label, inherited by every depth bin. Missing values and deployment gaps separate segments.

**Cycles.** Complete upward-zero-crossing cycles must satisfy your duration limits. They are interpolated
to a common normalized time and fitted as `offset + a cos(2πt) + b sin(2πt)`. R² is calculated from
residual sum of squares; it is not a Pearson correlation mislabeled as R².

**Profiles.** Phase-mean along-channel profiles and per-event peak profiles are retained separately.
Power laws fit the phase means at confirmed heights above bed. Log laws additionally require total
water depth and at least three bins within the chosen near-bed fraction. Unsupported fits are
reported as skipped; the near-bed range is not expanded automatically. The power exponent is
reported as α in `u = C z^(1/α)`; optional β uses the observed-bin mean and is not measured bed roughness.

**Resolved variability.** Subtract a trailing rolling mean before grouping residuals by phase/depth.
Slack remains in the detrending record; gaps/missing samples reset the window. A minimum of three
samples and 80% of a window are required. Six/ten-minute ensemble means cannot resolve the full
turbulent spectrum, Reynolds stresses, or a spectral Doppler-noise floor. The reported residual
intensity is therefore a coarse-record variability measure. Noise subtraction is optional and defaults
to zero; it is not a fitted spectral correction.

**Turbine settings and slack.** Set `cut_in_mps`, `rated_speed_mps`, `cut_out_mps`, and
`rated_kw` in JSON. Slack automatically equals cut-in, even when energy estimation is disabled.
Set `power_curve_mode` to `csv` to use `power_curve`, `edited` to use the JSON
`power_curve_points` array of `[speed_mps, power_kw]` pairs, or `generated` for a cubic rise
from cut-in to rated speed and a rated-power plateau up to cut-off. CSV/edited curves use
linear interpolation and retain their supplied shape. Rated speed shapes generated curves.
The effective curve is shown in the exported figure gallery. Settings and curve points are
preserved in the run log; the curve is also exported as CSV.

Velocity-profile panels place **ebb on the left with negative velocity** and **flood on the
right with positive velocity**. Peak speed-magnitude fits use the phase sign for display.

**Energy (optional).** A time-weighted histogram of the selected hub bin's absolute along-channel
velocity is combined with the supplied power curve. Cut-in/out are enforced; missing periods and
large gaps receive no invented energy. Annual energy is `8760 × availability × Σ(probability × power)`.
This extrapolates the observed distribution and is not a harmonic long-term prediction. Multiplying
by the turbine count gives a gross farm estimate without wake/electrical losses.

In [5]:
assessment = process(assessment)
show_results(assessment)

Floodward axis: 245.11° true. Positive along-channel flow = flood.


**Phase summary**

,phase,samples,observed_hours,observed_time_fraction,mean_speed_mps,peak_speed_mps,phase_axis_degT
0,FLOOD,3841,384.1,0.333884,0.692951,1.103064,244.489163
1,EBB,4068,406.8,0.353616,0.723274,1.183542,67.431677
2,SLACK,3594,359.4,0.312413,0.210453,0.434226,NaN
3,MISSING,1,0.1,0.000087,NaN,NaN,NaN


**Tidal-cycle fit**

,cycles,offset_mps,amplitude_mps,phase_shift_deg,r_squared,rmse_mps
0,91,-0.025726,0.857932,87.119703,0.997458,0.030549


...

**Energy estimate**

,hub_bin,valid_observed_hours,mean_power_kw,annual_kwh_per_turbine,observed_period_kwh,capacity_factor,turbines,availability,gross_farm_annual_kwh
0,13,1150.4,14.483658,126876.842837,16662.0,0.13167,24,1.0,3.045044e+06


**Original postprocessing checklist**

,original_workflow,postprocessing,table,figure,status
0,MATLAB / inputs02 cosine prototypes,Full-record fixed 12 h 25 min cosine and resid...,fixed_period_cosine_fit,_CosineFit,available
1,flowVisualized.py,Mean speeds and dominant-direction compass,dominant_directions,_FlowSummary,available
2,flowVisualized.py,Signed time series and velocity components,depth_averaged,_TimeSeries,available
3,flowVisualized.py,Phase histograms and magnitude box plots,phase_statistics,_Statistics,available
4,flowVisualized.py,Current rose,depth_averaged,current_rose,available
5,principal_flood_ebb_calculation.ipynb,"Ellipse, signed PDF and CDF",signed_velocity_distribution,principal_flow_diagnostics,available
6,principal_flood_ebb_calculation.ipynb,"Peaks, Reynolds range, operating durations",resource_metrics,(table only),available
7,tidal-cycles.ipynb,"Global, phase and event PCU summaries",pcu_summary,phase_pcu_comparison,available
8,tidal-cycles.ipynb,"Per-event axes, duration, RMS and peak PCU",tidal_events,event_diagnostics,available
9,tidal-cycles.ipynb,Normalized cycles and cosine view,cycle_original_fit,cycle_cosine_view,available


• CO-OPS timestamps retain the downloaded clock; its timezone is unspecified. Confirm it before external time comparisons.
• CO-OPS depth datum is unconfirmed. Depth profiles are available; physical log/power fits require a confirmed height above bed.
• 1 depth-averaged timestamps fail the selected bin-coverage requirement.
• Depth averages are arithmetic vector means over retained valid bins, not full-column integrals.
• Residual intensity is variability of the supplied ensembles. Six/ten-minute averages cannot resolve burst-scale turbulence or a Doppler-noise spectrum.
• AEP extrapolates this record's speed distribution to 8,760 hours. Gross farm scaling omits wakes, electrical losses and seasonal corrections.
• Original normalized profile fits use velocity-inferred bin coordinates, not confirmed physical z/h. Their alpha/beta and z0/h are retained as legacy diagnostics alongside the physical-coordinate fits.
• Original TI tables retain the horizontal-speed rolling denominator; the s

### Original postprocessing is retained

The **Original postprocessing checklist** maps every original analysis to a result table and
figure. Use the **Result table** selector above to inspect any complete analysis (the preview
shows 200 rows; exports contain all rows). The figure gallery below includes the six original
flow summaries, signed PDF/CDF and ellipse, event PCU statistics, cycle date/cosine views,
single-peak power profiles, event/log profiles, original TI comparisons, directional energy
probabilities, farm estimates, and the full-record 12 h 25 min cosine/residual fit.

Set `profile_coordinate_mode` in JSON to `legacy_normalized` or `physical`. Legacy normalized coordinates preserve the original fits but are not measured z/h.
The original TI tables preserve the rolling horizontal-speed denominator and are separate from
the newer residual-intensity tables. Optional annual-event comparisons use the annual input
folder; these prediction events never enter measured ADCP energy calculations. Verdant also
includes before/after heading diagnostics. Missing optional inputs and unsupported fits are
reported explicitly. See `backend/POSTPROCESSING_PARITY.md` for the detailed mapping.

## 3. Postprocess — save tables and figures

Each export creates matching, uniquely named run folders:

```text
outputs/
  csv/<site_source_run>/       Tables + run_log.json
  figures/<site_source_run>/   Publication-editable source data are in CSV; rendered figures are PNG
```

The run log records settings, source-file hashes, backend hashes, versions, coordinate/time metadata
and interpretation notes. Keep it with the CSVs when comparing sites or preparing paper results.
The saved tables include depth-resolved/averaged measurements, quality control, phases, cycles,
event-peak and phase-mean profiles, fit statistics, residual variability and energy-bin contributions.

In [6]:
export_locations = postprocess(assessment)
display(export_locations)

,result,folder
0,CSV tables and run log,C:\Users\hseyyedzadeh\Documents\Codex\papers\V...
1,Figures,C:\Users\hseyyedzadeh\Documents\Codex\papers\V...


### Explore the results

Select a depth bin, phase and portion of the record. These controls change the view only; they do
not recalculate or overwrite the assessment. The second selector opens the exported summary figures.

In [ ]:
explore(assessment)

### Run another source or change an assumption

Edit `config.json`, then rerun Setup and
Preprocess → Process → Postprocess. Preprocess reloads the file so saved changes are applied.
Prior outputs retain the exact effective settings in `run_log.json` for reproducibility.
For a custom site, update the source, input path and site-specific parameters in `config.json`.

**If a step stops:** read the exception message. Check the selected input path, site prefix, sheet names,
hub bin and units first. For missing profile fits, check depth datum and water depth; for no complete
cycles, check record duration and gaps. Do not remove quality checks just to obtain a figure.